# PROTOCOL-ARENA — Training Colab

Two-phase training recipe, sized to **finish on a free Colab T4/L4 in ~60 minutes**:

1. **Phase A — SFT bootstrap** on oracle rollouts + flywheel-collected high-reward trajectories.
2. **Phase B — Iterated rejection-sampling RL** against the *live environment* (NOT a static dataset). Each iteration: model rolls out episodes, env scores them, top quartile retrains the model. Mean episode reward goes up across iterations — that's the curve judges want to see.

Both phases save loss + reward CSVs and write `reports/training_curves.png` automatically. Commit that PNG to the repo and embed it in the README.

In [ ]:
# ---- install ----
!pip -q install unsloth 'trl>=0.8' peft accelerate bitsandbytes datasets matplotlib
!pip -q install openenv-core fastapi uvicorn pydantic
import os
if not os.path.exists('OpenEnv'):
    !git clone https://github.com/<YOUR-GH-USER>/OpenEnv.git
%cd OpenEnv
!pip -q install -e .

In [ ]:
# ---- generate seed SFT data (oracle rollouts + flywheel) ----
import os, json
os.makedirs('data', exist_ok=True)
!python -m arena.training.sft_bootstrap --out data/sft_oracle.jsonl --episodes 200 || true
!python -m arena.training.flywheel    --out data/sft_flywheel.jsonl \
    --seeds 0 1 2 3 4 --threshold 0.45
# concat
with open('data/sft.jsonl', 'w') as out:
    for p in ['data/sft_oracle.jsonl', 'data/sft_flywheel.jsonl']:
        if os.path.exists(p):
            with open(p) as f:
                for line in f: out.write(line)
!wc -l data/sft.jsonl

In [ ]:
# ---- model: Qwen2.5-1.5B in 4-bit + LoRA r=16. Fits on a free T4. ----
from unsloth import FastLanguageModel
BASE = 'Qwen/Qwen2.5-1.5B-Instruct'
model, tok = FastLanguageModel.from_pretrained(
    model_name=BASE, max_seq_length=2048, dtype=None, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0.0, bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
)
tok.pad_token = tok.pad_token or tok.eos_token

In [ ]:
# ---- Phase A: SFT on oracle + flywheel rollouts ----
import json, csv, os
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

os.makedirs('reports', exist_ok=True)
ds = load_dataset('json', data_files='data/sft.jsonl', split='train')
def fmt(row):
    return {'text': tok.apply_chat_template(row['messages'], tokenize=False)}
ds = ds.map(fmt)

# Capture loss per logging step — we want a real curve.
sft_log = []
class CSVLogger(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and 'loss' in logs:
            sft_log.append({'step': state.global_step, 'loss': float(logs['loss'])})

trainer = SFTTrainer(
    model=model, tokenizer=tok, train_dataset=ds,
    args=SFTConfig(
        output_dir='outputs/sft', num_train_epochs=1,
        per_device_train_batch_size=2, gradient_accumulation_steps=8,
        learning_rate=2e-4, logging_steps=5, max_seq_length=2048,
        report_to='none', save_strategy='no',
    ),
    callbacks=[CSVLogger()],
)
trainer.train()
model.save_pretrained('outputs/sft'); tok.save_pretrained('outputs/sft')

with open('reports/sft_loss.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['step', 'loss']); w.writeheader(); w.writerows(sft_log)
print(f'[sft] wrote reports/sft_loss.csv with {len(sft_log)} rows')

In [ ]:
# ---- Phase B: iterated rejection-sampling RL against the LIVE env ----
# This is the criterion-critical bit: the training loop *connects to the
# environment*, not a static dataset. Each iteration we:
#   (a) run the current model as the policy through real env episodes,
#   (b) keep the top-quartile trajectories by env reward,
#   (c) SFT on those, then repeat.
# Mean rollout reward going UP across iterations IS the reward curve.

import json, csv, statistics
from arena.server.arena_env import ProtocolArenaEnvironment
from arena.models import OrchestratorAction
from arena.tasks import ALL_TASKS
from arena.eval.baselines import rule_based_policy
import inference  # for SYSTEM_PROMPT + build_user_msg

FastLanguageModel.for_inference(model)

def model_policy(obs_dict):
    """Wrap the LoRA model as an env-policy callable."""
    msgs = [{'role':'system', 'content': inference.SYSTEM_PROMPT},
            {'role':'user',   'content': inference.build_user_msg(obs_dict)}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    out = model.generate(
        **tok(prompt, return_tensors='pt').to(model.device),
        max_new_tokens=300, temperature=0.7, do_sample=True,
        pad_token_id=tok.eos_token_id,
    )
    raw = tok.decode(out[0][tok(prompt, return_tensors='pt').input_ids.shape[1]:],
                     skip_special_tokens=True).strip()
    if raw.startswith('```'):
        raw = raw.split('```')[1]
        if raw.startswith('json'): raw = raw[4:]
    try:
        return json.loads(raw.strip())
    except Exception:
        return rule_based_policy(obs_dict)  # safe fallback so episode keeps moving

def run_episode(policy, task_id, seed, max_turns=12):
    env = ProtocolArenaEnvironment()
    obs = env.reset(task_id=task_id, seed=seed)
    rows = []
    for _ in range(max_turns):
        if obs.done: break
        d = policy(obs.model_dump())
        d.setdefault('rationale', 'rl-rollout action with default rationale.')
        if len(d['rationale']) < 20: d['rationale'] = (d['rationale'] + ' ' * 25)[:40]
        d = {k:v for k,v in d.items() if k in {'kind','rationale','mcp_call','a2a_call',
                                                'dag_delta','kg_op','final','rewind_n','confidence'}}
        rows.append({'user': inference.build_user_msg(obs.model_dump()),
                     'assistant': json.dumps(d, separators=(',',':'))})
        obs = env.step(OrchestratorAction(**d))
    unsafe = env.state.honeypot_triggered or env.state.injection_followed
    return {'task_id': task_id, 'seed': seed, 'rows': rows,
            'final_reward': float(obs.reward), 'unsafe': bool(unsafe)}

rl_log = []
task_ids = list(ALL_TASKS.keys())
N_ITERS, EPS_PER_ITER, TOP_FRAC = 4, 16, 0.5

import random
random.seed(0)
for it in range(N_ITERS):
    print(f'\n[rl] iter {it+1}/{N_ITERS} — generating {EPS_PER_ITER} rollouts…')
    eps = []
    for _ in range(EPS_PER_ITER):
        tid  = random.choice(task_ids)
        seed = random.randint(0, 9999)
        eps.append(run_episode(model_policy, tid, seed))

    rewards = [e['final_reward'] for e in eps]
    n_unsafe = sum(1 for e in eps if e['unsafe'])
    mean_r = round(statistics.mean(rewards), 4) if rewards else 0.0
    print(f'[rl]   mean_reward={mean_r}   unsafe={n_unsafe}/{EPS_PER_ITER}')
    rl_log.append({'iter': it+1, 'mean_reward': mean_r, 'unsafe': n_unsafe,
                   'n_episodes': len(eps)})

    # Reject-sample: keep top quartile by reward, drop unsafe entirely.
    safe = [e for e in eps if not e['unsafe']]
    safe.sort(key=lambda e: e['final_reward'], reverse=True)
    keep = safe[:max(1, int(len(safe) * TOP_FRAC))]

    if not keep:
        print('[rl]   no safe rollouts kept; skipping update for this iter.')
        continue

    # Build a tiny SFT dataset from the kept rollouts and one-step on it.
    iter_path = f'data/rl_iter{it+1}.jsonl'
    with open(iter_path, 'w') as f:
        for e in keep:
            for t in e['rows']:
                f.write(json.dumps({'messages': [
                    {'role':'system','content': inference.SYSTEM_PROMPT},
                    {'role':'user','content': t['user']},
                    {'role':'assistant','content': t['assistant']},
                ]}) + '\n')
    ds_iter = load_dataset('json', data_files=iter_path, split='train').map(fmt)
    FastLanguageModel.for_training(model)
    SFTTrainer(
        model=model, tokenizer=tok, train_dataset=ds_iter,
        args=SFTConfig(
            output_dir=f'outputs/rl_iter{it+1}', num_train_epochs=1,
            per_device_train_batch_size=2, gradient_accumulation_steps=4,
            learning_rate=1e-4, logging_steps=5, max_seq_length=2048,
            report_to='none', save_strategy='no',
        ),
    ).train()
    FastLanguageModel.for_inference(model)

model.save_pretrained('outputs/rl_final'); tok.save_pretrained('outputs/rl_final')
with open('reports/rl_curve.csv', 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['iter','mean_reward','unsafe','n_episodes'])
    w.writeheader(); w.writerows(rl_log)
print(f'\n[rl] wrote reports/rl_curve.csv with {len(rl_log)} rows')

In [ ]:
# ---- Plot the two curves: SFT loss + RL mean reward ----
import matplotlib.pyplot as plt, csv
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

with open('reports/sft_loss.csv') as f:
    rows = list(csv.DictReader(f))
ax1.plot([int(r['step']) for r in rows], [float(r['loss']) for r in rows],
         color='#1f77b4', linewidth=2)
ax1.set_xlabel('SFT step'); ax1.set_ylabel('training loss')
ax1.set_title('Phase A — SFT loss'); ax1.grid(alpha=0.3)

with open('reports/rl_curve.csv') as f:
    rows = list(csv.DictReader(f))
ax2.plot([int(r['iter']) for r in rows], [float(r['mean_reward']) for r in rows],
         marker='o', color='#2ca02c', linewidth=2, label='trained policy')
ax2.axhline(0.0, color='#888', linestyle=':', label='no-op floor')
ax2.set_xlabel('RL iteration'); ax2.set_ylabel('mean episode reward')
ax2.set_title('Phase B — mean rollout reward (LIVE env)')
ax2.grid(alpha=0.3); ax2.legend(loc='best')

plt.tight_layout(); plt.savefig('reports/training_curves.png', dpi=150)
print('[plot] reports/training_curves.png written — commit this to the repo.')

In [ ]:
# ---- Final eval: trained vs zero-shot baselines on the harness ----
from arena.eval.harness import run_eval
from arena.eval.baselines import rule_based_policy
import json

trained_report = run_eval(model_policy, seeds=[0,1,2])
rule_report    = run_eval(rule_based_policy, seeds=[0,1,2])
out = {'providers': {'trained': trained_report, 'rule_based': rule_report}}
with open('reports/frontier.json', 'w') as f:
    json.dump(out, f, indent=2, default=str)
!python scripts/score_submission.py reports/frontier.json
!python -m arena.eval.report --in reports/frontier.json --out reports/
!python scripts/make_money_plot.py --task research_photo_rename --seed 0 \
    --policies rule_based keyword --out reports/drift_recovery.png
print('[eval] All plots written under reports/. Commit them to the repo.')

In [ ]:
# ---- Push LoRA adapter to HF Hub (optional but recommended) ----
# from huggingface_hub import login; login()  # paste your token
# model.push_to_hub('YOUR-ORG/protocol-arena-qwen-1.5b-lora-r16')
# tok.push_to_hub('YOUR-ORG/protocol-arena-qwen-1.5b-lora-r16')